# RetainIQ — Phase 9.3: Retrieval Evaluation & Grounded Context

## Objective

I test retrieval quality against representative RetainIQ questions and build a ranked evidence context that can be passed to a generator. I treat the evaluation as a retrieval check, not as a substitute for human judgment of answer quality.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. The analytical code also prints a concise `Result:` line based on the executed data. Setup-only cells are not padded with artificial conclusions.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path('..')
OUTPUT_DIR = ROOT / 'outputs'
CONFIG_DIR = ROOT / 'config'

def print_result(message):
    print(f'Result: {message}')

chunks_df = pd.read_csv(OUTPUT_DIR / 'retrieval_chunks.csv')
vectors = np.load(OUTPUT_DIR / 'retrieval_vectors.npy')
manifest = json.loads((OUTPUT_DIR / 'retrieval_manifest.json').read_text(encoding='utf-8'))
questions = pd.read_csv(CONFIG_DIR / 'rag_sample_questions.csv')

print_result(f'Loaded {len(chunks_df):,} chunks and {len(questions):,} evaluation questions using the {manifest["backend"]} backend.')

Result: Loaded 8,414 chunks and 6 evaluation questions using the tfidf backend.


### Result & conclusion

The evaluation notebook is using the same saved index that will power the final RAG workflow, so retrieval tests are performed against the actual deployed artifact rather than a separate prototype.

In [4]:
def cosine_scores(query_vector, matrix):
    denom = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vector)
    denom = np.where(denom == 0, 1, denom)
    return (matrix @ query_vector) / denom

def vectorize_query(query):
    if manifest['backend'] == 'sentence_transformers':
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(manifest['embedding_model'])
        return np.asarray(model.encode([query], normalize_embeddings=True)[0], dtype=np.float32)
    import joblib
    vectorizer = joblib.load(OUTPUT_DIR / 'tfidf_vectorizer.joblib')
    return vectorizer.transform([query]).toarray()[0].astype(np.float32)

def retrieve(query, top_k=5):
    qv = vectorize_query(query)
    scores = cosine_scores(qv, vectors)
    order = np.argsort(-scores)[:top_k]
    result = chunks_df.iloc[order].copy()
    result['similarity'] = scores[order]
    result['rank'] = np.arange(1, len(result)+1)
    return result.reset_index(drop=True)

### Result & conclusion

The retrieval function now returns ranked evidence with a similarity score and a stable chunk identifier. The same function will be reused for answer generation.

In [3]:
def topic_relevance(row, expected_topics):
    return row.topic in expected_topics

evaluation_rows=[]
all_results=[]
for q in questions.itertuples(index=False):
    expected = {x.strip() for x in str(q.expected_topics).split('|') if x.strip()}
    retrieved = retrieve(q.question, top_k=5)
    relevant_ranks = [int(r.rank) for r in retrieved.itertuples(index=False) if topic_relevance(r, expected)]
    hit_at_5 = int(bool(relevant_ranks))
    reciprocal_rank = 1 / relevant_ranks[0] if relevant_ranks else 0.0
    evaluation_rows.append({
        'question_id': q.question_id,
        'question': q.question,
        'expected_topics': '|'.join(sorted(expected)),
        'hit_at_5': hit_at_5,
        'reciprocal_rank': reciprocal_rank,
        'top_source': retrieved.iloc[0]['source_name'] if len(retrieved) else None,
        'top_topic': retrieved.iloc[0]['topic'] if len(retrieved) else None,
    })
    view = retrieved.copy()
    view['question_id'] = q.question_id
    all_results.append(view)

evaluation = pd.DataFrame(evaluation_rows)
retrieval_results = pd.concat(all_results, ignore_index=True)
evaluation.to_csv(OUTPUT_DIR / 'retrieval_evaluation.csv', index=False)
retrieval_results.to_csv(OUTPUT_DIR / 'sample_retrieval_results.csv', index=False)

display(evaluation)
print_result(
    f"Hit@5 = {evaluation['hit_at_5'].mean():.1%}; "
    f"mean reciprocal rank = {evaluation['reciprocal_rank'].mean():.3f}."
)

,question_id,question,expected_topics,hit_at_5,reciprocal_rank,top_source,top_topic
0,Q01,What is the overall churn rate and how is it d...,benchmarking|definitions,1,1.000000,project_methodology,definitions
1,Q02,Which customer segment has the largest histori...,retention_strategy,1,0.333333,vw_benchmark_reference,benchmarking
2,Q03,How does a city's churn rate compare with its ...,benchmarking,1,0.250000,project_methodology,definitions
3,Q04,What does a 20 percent recovery scenario mean ...,definitions|retention_strategy,1,1.000000,vw_retention_action_framework,retention_strategy
4,Q05,What limitations should I mention when interpr...,definitions|geography,1,1.000000,project_methodology,definitions
5,Q06,How does the priority score incorporate ease o...,definitions|retention_strategy,1,1.000000,project_methodology,definitions


Result: Hit@5 = 100.0%; mean reciprocal rank = 0.764.


### Result & conclusion

This is a lightweight retrieval-quality check based on expected project topics. It tells me whether relevant evidence types are being surfaced; it does not replace human review of answer correctness.

In [4]:
selected_question = questions.iloc[0]['question']
context = retrieve(selected_question, top_k=5).copy()
context['evidence_id'] = [f'[{i}]' for i in range(1, len(context)+1)]

context_block = '\n\n'.join(
    f"{row.evidence_id} Source={row.source_name} | Topic={row.topic} | Similarity={row.similarity:.3f}\n{row.chunk_text}"
    for row in context.itertuples(index=False)
)

(OUTPUT_DIR / 'grounded_context_example.md').write_text(
    f"# Grounded context example\n\n## Question\n{selected_question}\n\n## Retrieved evidence\n{context_block}\n",
    encoding='utf-8'
)

display(context[['evidence_id','source_name','topic','similarity','chunk_text']])
print_result(f'Built a grounded context pack with {len(context)} evidence chunks for the sample question.')

,evidence_id,source_name,topic,similarity,chunk_text
0,[1],project_methodology,definitions,0.473903,GenAI / RAG Knowledge Layer ## Core definition...
1,[2],project_methodology,definitions,0.373325,chmark gap `Benchmark Gap = Observed Churn Rat...
2,[3],project_methodology,definitions,0.221735,# RetainIQ — Project Knowledge Reference ## Pr...
3,[4],vw_retention_action_framework,retention_strategy,0.112879,Source: vw_retention_action_framework | Topic:...
4,[5],vw_benchmark_reference,benchmarking,0.071428,Source: vw_benchmark_reference | Topic: benchm...


Result: Built a grounded context pack with 5 evidence chunks for the sample question.


### Result & conclusion

The grounded context pack is the handoff between retrieval and generation. The final RAG answer will be expected to rely only on this retrieved evidence plus the project's explicit interpretation rules.